## Document Ingestion

In [99]:
!pip install -q pymupdf


In [100]:
import fitz

pdf_path = "VQA_Visual_Question_Answering.pdf"

docs = fitz.open(pdf_path)

pages = []

for page_num, page in enumerate(docs):
  text = page.get_text()

  pages.append({
      "text": text,
      "page": page_num + 1,
  })
docs.close()

In [101]:
print(pages[0])
print(len(pages))
print(len(pages[0]['text']))

{'text': '1\nVQA: Visual Question Answering\nStanislaw Antol, Aishwarya Agrawal, Jiasen Lu, Margaret Mitchell,\nDhruv Batra, C. Lawrence Zitnick, Devi Parikh\nAbstract—We propose the task of free-form and open-ended Visual Question Answering (VQA). Given an image and a natural\nlanguage question about the image, the task is to provide an accurate natural language answer. Mirroring many real-world scenarios,\nsuch as helping the visually impaired, both the questions and answers are open-ended. Visual questions selectively target different\nareas of an image, including background details and underlying context. As a result, a system that succeeds at VQA typically needs a\nmore detailed understanding of the image and complex reasoning than a system producing generic image captions. Moreover, VQA is\namenable to automatic evaluation, since many open-ended answers contain only a few words or a closed set of answers that can be\nprovided in a multiple-choice format. We provide a dataset cont

## 2. COMBINE PAGES AND TRACK BOUNDARIES

In [102]:
full_text = ""
page_boundaries = []

for page in pages:

    start = len(full_text)

    full_text += page["text"] + "\n"

    end = len(full_text)

    page_boundaries.append({
        "page": page["page"],
        "start": start,
        "end": end
    })

# Tokenization

In [103]:
!pip install -q tiktoken

In [104]:
# import tiktoken and encoding "cl100k_base"
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

In [105]:
# Assumes enc is an initialized tokenizer
token_ids = enc.encode(full_text)

# Get character offsets for each token
token_offsets = enc.decode_with_offsets(token_ids)[1]

## Chunking

In [106]:
chunk_size = 200
overlap = 30
step = chunk_size - overlap

chunks = []

for start in range(0, len(token_ids), step):

    end = min(start + chunk_size, len(token_ids))

    chunk_tokens = token_ids[start:end]

    if not chunk_tokens:
        continue

    chunk_text = enc.decode(chunk_tokens)

    # Map token positions to character positions
    chunk_start = token_offsets[start]

    chunk_end = (
        token_offsets[end]
        if end < len(token_offsets)
        else len(full_text)
    )

    # Find all pages overlapping this chunk
    chunk_pages = []

    for boundary in page_boundaries:

        if (
            chunk_start < boundary["end"]
            and chunk_end > boundary["start"]
        ):
            chunk_pages.append(boundary["page"])

    chunks.append({
        "chunk_id": len(chunks),
        "text": chunk_text,
        "token_ids": chunk_tokens,
        "metadata": {
            "source": pdf_path,
            "pages": chunk_pages
        }
    })

In [107]:
print("Total pages:", len(pages))
print("Total tokens:", len(token_ids))
print("Total chunks:", len(chunks))

for chunk in chunks[:5]:

    print("\n" + "=" * 60)

    print("Chunk ID:", chunk["chunk_id"])
    print("Pages:", chunk["metadata"]["pages"])
    print("Token count:", len(chunk["token_ids"]))
    print("Text:", chunk["text"])

Total pages: 16
Total tokens: 21263
Total chunks: 126

Chunk ID: 0
Pages: [1]
Token count: 200
Text: 1
VQA: Visual Question Answering
Stanislaw Antol, Aishwarya Agrawal, Jiasen Lu, Margaret Mitchell,
Dhruv Batra, C. Lawrence Zitnick, Devi Parikh
Abstract—We propose the task of free-form and open-ended Visual Question Answering (VQA). Given an image and a natural
language question about the image, the task is to provide an accurate natural language answer. Mirroring many real-world scenarios,
such as helping the visually impaired, both the questions and answers are open-ended. Visual questions selectively target different
areas of an image, including background details and underlying context. As a result, a system that succeeds at VQA typically needs a
more detailed understanding of the image and complex reasoning than a system producing generic image captions. Moreover, VQA is
amenable to automatic evaluation, since many open-ended answers contain only a few words or a closed set of an

## Vector Embedding

In [108]:
!pip install -q sentence-transformers

In [109]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [110]:
texts = [chunk['text'] for chunk in chunks]
embeddings = model.encode(texts)

In [111]:
print(type(embeddings))
print(embeddings.shape)
print(len(chunks))
print(embeddings)

<class 'numpy.ndarray'>
(126, 384)
126
[[ 0.00814298  0.03054151 -0.01123812 ...  0.03585466  0.02966289
  -0.01235218]
 [ 0.05975164 -0.02467837 -0.04181396 ...  0.00328315  0.00937819
  -0.01901968]
 [ 0.0188289  -0.05292328 -0.00871951 ...  0.00933112 -0.03633752
  -0.03479125]
 ...
 [-0.01451151  0.00471448  0.06388738 ...  0.0842725  -0.02066324
   0.00566111]
 [ 0.01619266  0.0019314   0.0244375  ...  0.08429962 -0.00597607
  -0.02866727]
 [-0.07323704  0.01888612  0.01332539 ... -0.11002208 -0.04244973
   0.04366078]]


## Storing in FAISS

In [112]:
!pip install -q faiss-cpu

In [113]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print(index.ntotal)

126


## Retrieval

In [114]:
def retrieve(query, k=3):

    query_embedding = model.encode([query])

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    distances, indices = index.search(query_embedding, k)

    results = []

    for i, idx in enumerate(indices[0]):

        results.append({
            "chunk": chunks[idx],
            "distance": float(distances[0][i])
        })

    return results

In [115]:
results = retrieve(
    "What is Visual Question Answering",
    k=3
)

for result in results:

    print("Distance:", result["distance"])
    print("Text:", result["chunk"]["text"])
    print("Pages:", result["chunk"]["metadata"]["pages"])
    print("-" * 50)

Distance: 0.5076826810836792
Text: 1
VQA: Visual Question Answering
Stanislaw Antol, Aishwarya Agrawal, Jiasen Lu, Margaret Mitchell,
Dhruv Batra, C. Lawrence Zitnick, Devi Parikh
Abstract—We propose the task of free-form and open-ended Visual Question Answering (VQA). Given an image and a natural
language question about the image, the task is to provide an accurate natural language answer. Mirroring many real-world scenarios,
such as helping the visually impaired, both the questions and answers are open-ended. Visual questions selectively target different
areas of an image, including background details and underlying context. As a result, a system that succeeds at VQA typically needs a
more detailed understanding of the image and complex reasoning than a system producing generic image captions. Moreover, VQA is
amenable to automatic evaluation, since many open-ended answers contain only a few words or a closed set of answers that can be

Pages: [1]
------------------------------------

## Augmentation

In [116]:
context_parts = []

for i, result in enumerate(results):

    chunk = result["chunk"]

    chunk_context = f"""
    [Chunk {i+1}]
    [Pages: {chunk['metadata']['pages']}]

    {chunk['text']}
    """

    context_parts.append(chunk_context)

context = "\n\n".join(context_parts)

In [117]:
context

'\n    [Chunk 1]\n    [Pages: [1]]\n\n    1\nVQA: Visual Question Answering\nStanislaw Antol, Aishwarya Agrawal, Jiasen Lu, Margaret Mitchell,\nDhruv Batra, C. Lawrence Zitnick, Devi Parikh\nAbstract—We propose the task of free-form and open-ended Visual Question Answering (VQA). Given an image and a natural\nlanguage question about the image, the task is to provide an accurate natural language answer. Mirroring many real-world scenarios,\nsuch as helping the visually impaired, both the questions and answers are open-ended. Visual questions selectively target different\nareas of an image, including background details and underlying context. As a result, a system that succeeds at VQA typically needs a\nmore detailed understanding of the image and complex reasoning than a system producing generic image captions. Moreover, VQA is\namenable to automatic evaluation, since many open-ended answers contain only a few words or a closed set of answers that can be\n\n    \n\n\n    [Chunk 2]\n    

In [118]:
query = "What is VQA?"

In [119]:
prompt = f"""
Answer the question using the provided context.

If the answer cannot be found in the context,
say that you don't know based on the provided document.

Context:
{context}

Question:
{query}

Answer:
"""

In [120]:
print(prompt)


Answer the question using the provided context.

If the answer cannot be found in the context,
say that you don't know based on the provided document.

Context:

    [Chunk 1]
    [Pages: [1]]

    1
VQA: Visual Question Answering
Stanislaw Antol, Aishwarya Agrawal, Jiasen Lu, Margaret Mitchell,
Dhruv Batra, C. Lawrence Zitnick, Devi Parikh
Abstract—We propose the task of free-form and open-ended Visual Question Answering (VQA). Given an image and a natural
language question about the image, the task is to provide an accurate natural language answer. Mirroring many real-world scenarios,
such as helping the visually impaired, both the questions and answers are open-ended. Visual questions selectively target different
areas of an image, including background details and underlying context. As a result, a system that succeeds at VQA typically needs a
more detailed understanding of the image and complex reasoning than a system producing generic image captions. Moreover, VQA is
amenable to 

## LLM Implementation

In [121]:
!pip install -q groq

In [122]:
import os
os.environ['GROQ_API_KEY'] = "<Your api key>"

In [123]:
from groq import Groq

llm = Groq(
     api_key=os.environ.get("GROQ_API_KEY")
)

In [124]:
def generate_answer(prompt):

  response = llm.chat.completions.create(
      model='openai/gpt-oss-20b',

      messages=[
          {
              "role": "system",
              "content": "You are a helpful assistant. Answer using the provided context."
          },
          {
              "role": "user",
              "content": prompt
          }
      ],

      temperature=0.2,
      max_completion_tokens=1000
  )

  print("Finish reason:", response.choices[0].finish_reason)
  print("Usage:", response.usage)

  return response.choices[0].message.content

## Improve Prompt and Include Sources

In [125]:
test_queries = [
    "What is Visual Question Answering?",
    "What datasets are used for VQA?",
    "What are the challenges of Visual Question Answering?"
]

for query in test_queries:

    results = retrieve(query, k=3)

    sources = []

    context_parts = []

    for i, result in enumerate(results):

        chunk = result["chunk"]

        source = chunk["metadata"]["source"]
        pages = chunk["metadata"]["pages"]

        chunk_context = f"""
        [Chunk {i+1}]
        [Source: {source}]
        [Pages: {pages}]

        {chunk["text"]}
        """

        context_parts.append(chunk_context)
        sources.append({
        "source": chunk["metadata"]["source"],
        "pages": chunk["metadata"]["pages"],
        "chunk_id": chunk["chunk_id"]
    })

    context = "\n\n".join(context_parts)

    prompt = f"""
    Answer the question using only the provided context.

    Instructions:
    1. Use the context to answer the question.
    2. Cite the relevant page numbers using [Page X].
    3. If information comes from multiple pages, cite all relevant pages.
    4. Do not invent information or page numbers.
    5. If the answer is unavailable, say you don't know.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    answer = generate_answer(prompt)

    print("Question:", query)
    print("Answer:", answer)
    print("-" * 80)

    print("Sources:")

    for source in sources:
        print(
            f"File: {source['source']} | "
            f"Pages: {source['pages']} | "
            f"Chunk ID: {source['chunk_id']}"
        )

    print("-" * 80)

Finish reason: stop
Usage: CompletionUsage(completion_tokens=236, prompt_tokens=870, total_tokens=1106, completion_time=0.251897889, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=158), prompt_time=0.046063264, prompt_tokens_details=None, queue_time=0.075002458, total_time=0.297961153)
Question: What is Visual Question Answering?
Answer: Visual Question Answering (VQA) is a task in which a system is given an image and a natural‑language question about that image, and it must produce an accurate natural‑language answer. The task is free‑form and open‑ended, requiring deeper image understanding and reasoning beyond what generic image captions provide. [Page 1]
--------------------------------------------------------------------------------
Sources:
File: VQA_Visual_Question_Answering.pdf | Pages: [1] | Chunk ID: 0
File: VQA_Visual_Question_Answering.pdf | Pages: [7] | Chunk ID: 46
File: VQA_Visual_Question_Answering.pdf | Pages: [2] | Chunk ID: 14
--------------------

## Evaluation using Precision@K, Recall@K and MRR

In [151]:
for chunk in chunks[5:15]:

    print("Chunk ID:", chunk["chunk_id"])
    print("Pages:", chunk["metadata"]["pages"])
    print(chunk["text"])
    print("=" * 100)

Chunk ID: 5
Pages: [1]
images via Amazon Mechanical Turk. Note that commonsense
knowledge is needed along with a visual understanding of the scene
to answer many questions.
have 20/20 vision?”, “Is this person expecting company?”).
VQA [16], [30], [40], [2] is also amenable to automatic
quantitative evaluation, making it possible to effectively track
progress on this task. While the answer to many questions is
simply “yes” or “no”, the process for determining a correct
answer is typically far from trivial (for instance in Fig. 1,
“Does this person have 20/20 vision?”). Moreover, since
questions about images often tend to seek speciﬁc information,
simple one-to-three word answers are sufﬁcient for many
questions. In such scenarios, we can easily evaluate a proposed
algorithm by the number of questions it answers correctly. In
this paper, we present both an open-ended answering task
Chunk ID: 6
Pages: [1, 2]
 we can easily evaluate a proposed
algorithm by the number of questions it answe

In [171]:
ground_truth = {

    "What is Visual Question Answering?": [
        0,3
    ],

    "What datasets are used for VQA?": [
        6, 7, 11, 27
    ],

    "What are the challenges of Visual Question Answering?": [
        0, 8, 53
    ]

}

In [172]:
retrieval_results = {}

for query in ground_truth:

    results = retrieve(query, k=3)

    retrieved_ids = [
        result["chunk"]["chunk_id"]
        for result in results
    ]

    retrieval_results[query] = retrieved_ids

    print("Query:", query)
    print("Retrieved chunks:", retrieved_ids)
    print("Relevant chunks:", ground_truth[query])
    print("-" * 60)

Query: What is Visual Question Answering?
Retrieved chunks: [0, 46, 14]
Relevant chunks: [0, 3]
------------------------------------------------------------
Query: What datasets are used for VQA?
Retrieved chunks: [11, 27, 47]
Relevant chunks: [6, 7, 11, 27]
------------------------------------------------------------
Query: What are the challenges of Visual Question Answering?
Retrieved chunks: [0, 3, 53]
Relevant chunks: [0, 8, 53]
------------------------------------------------------------


In [173]:
def precision_at_k(retrieved_ids, relevant_ids, k):

    retrieved = retrieved_ids[:k]

    relevant_retrieved = set(retrieved) & set(relevant_ids)

    precision = len(relevant_retrieved) / k

    return precision

In [174]:
def recall_at_k(retrieved_ids, relevant_ids, k):

    retrieved = retrieved_ids[:k]

    relevant_retrieved = set(retrieved) & set(relevant_ids)

    recall = len(relevant_retrieved) / len(relevant_ids)

    return recall

In [175]:
def reciprocal_rank(retrieved_ids, relevant_ids):

    for rank, chunk_id in enumerate(retrieved_ids, start=1):

        if chunk_id in relevant_ids:
            return 1 / rank

    return 0

In [176]:
def mean_reciprocal_rank(retrieval_results, ground_truth):

    reciprocal_ranks = []

    for query in ground_truth:

        retrieved_ids = retrieval_results[query]
        relevant_ids = ground_truth[query]

        rr = reciprocal_rank(
            retrieved_ids,
            relevant_ids
        )

        reciprocal_ranks.append(rr)

    return sum(reciprocal_ranks) / len(reciprocal_ranks)

In [177]:
k = 3

for query in ground_truth:

    retrieved_ids = retrieval_results[query]
    relevant_ids = ground_truth[query]

    precision = precision_at_k(
        retrieved_ids, relevant_ids, k
    )

    recall = recall_at_k(
        retrieved_ids, relevant_ids, k
    )

    print("Query:", query)
    print("Precision@3:", round(precision, 3))
    print("Recall@3:", round(recall, 3))
    print("-" * 60)

Query: What is Visual Question Answering?
Precision@3: 0.333
Recall@3: 0.5
------------------------------------------------------------
Query: What datasets are used for VQA?
Precision@3: 0.667
Recall@3: 0.5
------------------------------------------------------------
Query: What are the challenges of Visual Question Answering?
Precision@3: 0.667
Recall@3: 0.667
------------------------------------------------------------


In [178]:
mrr = mean_reciprocal_rank(
    retrieval_results,
    ground_truth
)

print("MRR:", round(mrr, 3))

MRR: 1.0


## K value Comparison [1, 3, 5]

In [179]:
retrieval_results_k5 = {}

for query in ground_truth:

    results = retrieve(query, k=5)

    retrieved_ids = [
        result["chunk"]["chunk_id"]
        for result in results
    ]

    retrieval_results_k5[query] = retrieved_ids

    print("Query:", query)
    print("Retrieved chunks:", retrieved_ids)
    print("-" * 60)

Query: What is Visual Question Answering?
Retrieved chunks: [0, 46, 14, 3, 9]
------------------------------------------------------------
Query: What datasets are used for VQA?
Retrieved chunks: [11, 27, 47, 15, 46]
------------------------------------------------------------
Query: What are the challenges of Visual Question Answering?
Retrieved chunks: [0, 3, 53, 9, 46]
------------------------------------------------------------


In [180]:
for query in ground_truth:

    retrieved_ids = retrieval_results_k5[query]
    relevant_ids = ground_truth[query]

    precision = precision_at_k(
        retrieved_ids, relevant_ids, 5
    )

    recall = recall_at_k(
        retrieved_ids, relevant_ids, 5
    )

    print("Query:", query)
    print("Precision@5:", round(precision, 3))
    print("Recall@5:", round(recall, 3))
    print("-" * 60)

Query: What is Visual Question Answering?
Precision@5: 0.4
Recall@5: 1.0
------------------------------------------------------------
Query: What datasets are used for VQA?
Precision@5: 0.4
Recall@5: 0.5
------------------------------------------------------------
Query: What are the challenges of Visual Question Answering?
Precision@5: 0.4
Recall@5: 0.667
------------------------------------------------------------


In [181]:
mrr_k5 = mean_reciprocal_rank(
    retrieval_results_k5,
    ground_truth
)

print("MRR@5:", round(mrr_k5, 3))

MRR@5: 1.0


In [182]:
retrieval_results_k1 = {}

for query in ground_truth:

    results = retrieve(query, k=1)

    retrieved_ids = [
        result["chunk"]["chunk_id"]
        for result in results
    ]

    retrieval_results_k1[query] = retrieved_ids

    print("Query:", query)
    print("Retrieved chunks:", retrieved_ids)
    print("-" * 60)

Query: What is Visual Question Answering?
Retrieved chunks: [0]
------------------------------------------------------------
Query: What datasets are used for VQA?
Retrieved chunks: [11]
------------------------------------------------------------
Query: What are the challenges of Visual Question Answering?
Retrieved chunks: [0]
------------------------------------------------------------


In [183]:
for query in ground_truth:

    retrieved_ids = retrieval_results_k1[query]
    relevant_ids = ground_truth[query]

    precision = precision_at_k(
        retrieved_ids, relevant_ids, 1
    )

    recall = recall_at_k(
        retrieved_ids, relevant_ids, 1
    )

    print("Query:", query)
    print("Precision@1:", round(precision, 3))
    print("Recall@1:", round(recall, 3))
    print("-" * 60)

Query: What is Visual Question Answering?
Precision@1: 1.0
Recall@1: 0.5
------------------------------------------------------------
Query: What datasets are used for VQA?
Precision@1: 1.0
Recall@1: 0.25
------------------------------------------------------------
Query: What are the challenges of Visual Question Answering?
Precision@1: 1.0
Recall@1: 0.333
------------------------------------------------------------


In [184]:
mrr_k1 = mean_reciprocal_rank(
    retrieval_results_k1,
    ground_truth
)

print("MRR@1:", round(mrr_k1, 3))

MRR@1: 1.0


In [185]:
def evaluate_retriever(ground_truth, k_values):

    evaluation_results = {}

    for k in k_values:

        retrieval_results = {}

        # Retrieve chunks for every query
        for query in ground_truth:

            results = retrieve(query, k=k)

            retrieved_ids = [
                result["chunk"]["chunk_id"]
                for result in results
            ]

            retrieval_results[query] = retrieved_ids

        # Calculate metrics
        precisions = []
        recalls = []

        for query in ground_truth:

            retrieved_ids = retrieval_results[query]
            relevant_ids = ground_truth[query]

            precisions.append(
                precision_at_k(retrieved_ids, relevant_ids, k)
            )

            recalls.append(
                recall_at_k(retrieved_ids, relevant_ids, k)
            )

        mrr = mean_reciprocal_rank(
            retrieval_results,
            ground_truth
        )

        evaluation_results[k] = {
            "precision": sum(precisions) / len(precisions),
            "recall": sum(recalls) / len(recalls),
            "mrr": mrr
        }

    return evaluation_results

In [186]:
k_values = [1, 3, 5]

evaluation_results = evaluate_retriever(
    ground_truth,
    k_values
)

In [187]:
for k, metrics in evaluation_results.items():

    print(f"\nEvaluation for K={k}")

    print("Precision:", round(metrics["precision"], 3))
    print("Recall:", round(metrics["recall"], 3))
    print("MRR:", round(metrics["mrr"], 3))


Evaluation for K=1
Precision: 1.0
Recall: 0.361
MRR: 1.0

Evaluation for K=3
Precision: 0.556
Recall: 0.556
MRR: 1.0

Evaluation for K=5
Precision: 0.4
Recall: 0.722
MRR: 1.0
